# iPhone head benchmark and encoder audit
Mac/Xcode + kernel SongFormer Python 3.11. Membuat proyek Xcode terpisah di
outputs.local; tidak mengubah SongMap atau memicu TestFlight. Model hanya head:
FP32, bobot INT8 dan palette6. Timing bukan end-to-end audio. Awali fixture30s,
bukan 420s, untuk menguji kelayakan perangkat. Sumber dipilih dari hasil kompresi
terbaru; periksa path yang dicetak. Tidak ada model atau audio diunggah.


In [ ]:
import subprocess, sys
from pathlib import Path
from datetime import datetime
assert Path('finetuning/config.json').is_file()
SONGMAP_ROOT = Path('/Users/danendra/Documents/ADA26/C6/SongMap')
comparisons = sorted(Path('outputs.local').glob('coreml-compression-*/comparison.json'),
                     key=lambda p: p.stat().st_mtime, reverse=True)
assert comparisons, 'Laporan kompresi belum ditemukan'
comparison = comparisons[0]
output = Path('outputs.local') / ('iphone-benchmark-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
print('Compression comparison:', comparison)

def run_logged(command, log_path):
    with log_path.open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if code:
        raise RuntimeError(f'Proses gagal ({code}); lihat {log_path}')


Persiapkan proyek dan fixture nyata30s identik. Menjalankan coremlcompiler dari Xcode, tidak melatih model.

In [ ]:
run_logged([sys.executable, '-u', '-m', 'deployment.prepare_iphone',
    '--comparison', str(comparison), '--output-dir', str(output),
    '--songmap-root', str(SONGMAP_ROOT), '--seconds', '30'],
    Path(str(output) + '.prepare.log'))
print('Project:', output / 'Benchmark.xcodeproj')


Buka proyek Xcode, pilih target Benchmark -> Signing & Capabilities -> Team
milik Anda. Jika perlu ubah bundle identifier agar unik untuk akun Anda.
Pilih iPhone17 fisik sebagai run destination, lalu Run. Aktifkan Developer Mode
jika diminta iPhone. Ini memerlukan pengaturan signing Xcode pribadi; tidak perlu
mengubah signing/repository SongMap. Tombol app menguji tiga kandidat berurutan.
Uji mode all dahulu, lalu cpuAndNeuralEngine. Setelah selesai gunakan Share JSON
report untuk AirDrop/simpan ke Mac. Simulator bukan pengukuran performa iPhone.
Untuk perbandingan stabil gunakan konfigurasi daya sama, tunggu perangkat dingin,
dan ulang beberapa kali. Waktu first prediction tidak menjamin cold OS cache.


In [ ]:
subprocess.run(['open', str(output / 'Benchmark.xcodeproj')], check=True)


Audit encoder pada Mac (terpisah dari benchmark head). Membaca audio pertama val, bukan seluruh dataset. Cache tidak ditulis ulang. Tunggu setiap window selesai.

In [ ]:
for seconds in (30, 120):
    run_logged([sys.executable, '-u', '-m', 'deployment.verify_encoders',
        '--songmap-root', str(SONGMAP_ROOT), '--seconds', str(seconds),
        '--output', str(output / f'encoders-{seconds}s.json')],
        output / f'encoders-{seconds}s.log')


In [ ]:
import json
for seconds in (30, 120):
    report = json.loads((output / f'encoders-{seconds}s.json').read_text())
    print('Window:', seconds, 's')
    for result in report['results']:
        print(json.dumps(result, indent=2))
